<a href="https://colab.research.google.com/github/Juliodelemos/sugarcane-variety-classification-yolov8/blob/main/recognition_4_varieties_YOLOv8_CLASSIFY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print(f"GPU disponível: {torch.cuda.is_available()}")
print(f"Número de GPUs: {torch.cuda.device_count()}")

In [ ]:
# IMPORTANT: Follow the instructions in the markdown cell above before running this code.
# This block evaluates the model on the TEST set by temporarily treating it as the validation set.

# IMPORTANT: Update these paths to your trained model, dataset, and desired project output location.
# The 'data' argument can point to the data.yaml file if it correctly specifies the 'val' (now test) path.
model_path = './runs/classify/my_classification_experiment/val_results/weights/best.pt' # Path to your best trained model
dataset_path = './data.yaml' # Path to your data.yaml file, which should point to the (renamed) 'val' folder for testing
project_path = './runs/classify/my_test_evaluation_project' # Project path for test evaluation results
experiment_name = 'test_results_run'

metrics = evaluate_model(model_path, dataset_path, project_path, experiment_name)
metrics_model(metrics=metrics)

# IMPORTANT: After running, remember to revert the directory names as described in the markdown cell above.

In [ ]:
# Connect Colab to Google Drive (if datasets are stored there)
# Reviewers may need to adjust this or replace paths if using local storage or other cloud storage.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics # Instalar a biblioteca Ultralytics (YOLOv8)
!pip install tensorboard

# Image Distribution

# Imports

In [ ]:
import os
from ultralytics import YOLO
from collections import Counter
from pathlib import Path

# Helper Functions

In [ ]:
def contaImgArqDir(organized_path):
  # Contagem de imagens por classe
  image_counts = Counter()
  for root, _, files in os.walk(organized_path):
      for file in files:
          if file.endswith(('.png', '.jpg','.JPG', '.jpeg')):
              class_name = Path(root).name  # Nome da pasta indica a classe
              image_counts[class_name] += 1

  print(f"\nResumo da contagem:{organized_path}")
  for dir_name, count in image_counts.items():
      print(f"Diretório '{dir_name}': {count} imagens")

In [ ]:
# IMPORTANT: Update this path to your original cropped dataset directory.
contaImgArqDir("./data_cropped")

# Folder Structure

In [ ]:
from sklearn.model_selection import train_test_split
import shutil
from pathlib import Path
import os

# Base paths for the cropped dataset and for the split datasets (train, val, test)
# IMPORTANT: Reviewers should update these paths to their local dataset locations.
base_dir = Path('./data_cropped') # Insert here the path to the original cropped dataset

train_dir = Path('./datasets/train') # Path for training data output
val_dir = Path('./datasets/val')   # Path for validation data output
test_dir = Path('./datasets/test')  # Path for test data output

def split_dataset_multiclass(base_path, train_path, val_path, test_path, test_size=0.1, val_size=0.1, random_state=42):
    """
    Splits the dataset into training, validation, and test sets, organized by classes.
    Args:
        base_path (Path): Path to the base dataset directory containing class subfolders.
        train_path (Path): Path to save the training data.
        val_path (Path): Path to save the validation data.
        test_path (Path): Path to save the test data.
        test_size (float): Proportion of data reserved for the test set.
        val_size (float): Proportion of data reserved for the validation set.
        random_state (int): Seed for reproducible splitting.
    """
    if not base_path.exists():
        raise FileNotFoundError(f"The base directory {base_path} does not exist.")

    # Creating target directories
    for path in [train_path, val_path, test_path]:
        if path.exists():
            shutil.rmtree(path)
        path.mkdir(parents=True)

    # Available classes. Update this list if your dataset has different classes.
    classes = ["CTC_1007","IAC_5503", "RB_5375", "RB_7515"]

    for class_name in classes:
        images = []
        # Collecting images for the current class
        class_dir = base_path / class_name
        if class_dir.exists():
            images.extend(list(class_dir.glob("*.JPG")) + list(class_dir.glob("*.png")))
        else:
            print(f"Subdirectory {class_name} not found. Skipping.")
            continue

        if len(images) == 0:
            print(f"No images available for class '{class_name}'.")
            continue

        # Splitting into train, validation, and test sets
        train_images, temp_images = train_test_split(images, test_size=(test_size + val_size), random_state=random_state)
        val_images, test_images = train_test_split(temp_images, test_size=test_size / (test_size + val_size), random_state=random_state)

        # Copying files to their respective destination directories
        for img_paths, dest_dir in zip([train_images, val_images, test_images], [train_path, val_path, test_path]):
            class_dest_dir = dest_dir / class_name
            class_dest_dir.mkdir(exist_ok=True)
            for img in img_paths:
                shutil.copy(img, class_dest_dir / img.name)

        # Displaying image counts for each split
        print(f"{class_name}:")
        print(f" - Training: {len(train_images)}")
        print(f" - Validation: {len(val_images)}")
        print(f" - Test: {len(test_images)}")

    print("Dataset splitting completed successfully.")

    # Verifying if image counts are correct after splitting
    for class_name in classes:
        # Counting images in the original directory
        original_count = len(list((base_path / class_name).glob("*.JPG"))) + len(list((base_path / class_name).glob("*.png")))
        # Counting images in the split directories (train, val, test)
        split_count = sum(len(list((split_path / class_name).glob("*.JPG"))) + len(list((split_path / class_name).glob("*.png")))
                          for split_path in [train_path, val_path, test_path])
        print(f"\nVerification for class '{class_name}':")
        print(f" - Total in original directory: {original_count}")
        print(f" - Total in split directories: {split_count}")
        if original_count == split_count:
            print(f"✅ Image count is correct for class '{class_name}'.")
        else:
            print(f"❌ Image count is INCORRECT for class '{class_name}'.")

# Splitting the dataset
split_dataset_multiclass(base_dir, train_dir, val_dir, test_dir)

# Transfer Learning Training

In [ ]:
# Importing necessary libraries
from sklearn.model_selection import train_test_split
import shutil
from pathlib import Path
import os
import yaml  # Import the YAML library

# Defining base paths for the directories
# IMPORTANT: Reviewers should update these paths to their local dataset locations.
dataset_path = Path("./datasets") # Directory where train, val, and test subdirectories are located.
train_dir = dataset_path / "train"  # Path to training data
val_dir = dataset_path / "val"  # Path to validation data
test_dir = dataset_path / "test"  # Path to test data
data_yaml_path = "./data.yaml" # Path for the data.yaml file


In [ ]:
# Creating the data.yaml file

data = {
    "train": str(train_dir),
    "val": str(val_dir),
    "test": str(test_dir),
    "nc": len(os.listdir(train_dir)),  # Number of classes
    "names": {i: name for i, name in enumerate(os.listdir(train_dir))},  # Class names
}

# YAML without the test path (commented out as per original notebook's intent for a full data.yaml)
'''
data = {
    "train": str(train_dir),
    "val": str(val_dir),
    "nc": len(os.listdir(train_dir)),  # Number of classes
    "names": {i: name for i, name in enumerate(os.listdir(train_dir))},  # Class names
}
'''

# Saving the data.yaml file
with open(data_yaml_path, "w") as f:
    yaml.dump(data, f)

print(f"data.yaml file saved to: {data_yaml_path}")

In [ ]:
data_yaml_path = "/content/drive/MyDrive/Colab_Notebooks/four_varieties/datasets" # Caminho para as pastas Train, Val e Test

In [ ]:
from ultralytics import YOLO

# Load a pre-trained model (YOLOv8n-cls)
model = YOLO('yolov8n-cls.pt')

# Define paths and hyperparameters
# IMPORTANT: Reviewers should specify a project path where training results will be saved.
project_path = './runs/classify/my_classification_experiment' # Path for saving project results
experiment_name = 'val_results'
num_epochs = 30

# Training with custom hyperparameters
results = model.train(
    task='classify',
    mode='train',
    data=data_yaml_path,  # Path to the data.yaml file
    epochs=num_epochs,  # Maximum number of epochs
    batch=16,  # Batch size
    imgsz=224,  # Image size
    device='0',  # Use 'cpu' or '0' if GPU is available
    workers=2,  # Number of threads for data loading
    hsv_h=0.3,  # Data augmentation: hue variation (HSV-H)
    hsv_s=0.6,  # Data augmentation: saturation variation (HSV-S)
    hsv_v=0.4,  # Data augmentation: value variation (HSV-V)
    degrees=10,  # Random rotation (degrees)
    translate=0.1,  # Horizontal and vertical translation
    scale=0.1,  # Random scaling
    shear=5,  # Random shearing
    fliplr=0.5,  # Horizontal flip with 50% probability
   # flipud=0.2,  # Vertical flip with 20% probability (commented out in original)
    mixup=0.1,  # Mixup augmentation
    optimizer='AdamW',  # Optimizer
    lr0=0.001,  # Initial learning rate
    weight_decay=0.0001,  # L2 regularization
    seed=42,  # Seed for reproducibility
    patience=10,  # Early stopping: stop after 10 epochs without validation improvement
    project=project_path,  # Project name
    name=experiment_name  # Experiment name
)

In [ ]:
print(model.names)

# Start TensorBoard

In [ ]:
!pip install netron

In [ ]:
import netron
#netron.start("/content/drive/MyDrive/Colab_Notebooks/four_varieties/four_cropped_varieties_artigo_80_20/val_results-2/weights/best.onnx")
netron.start("/content/drive/MyDrive/Colab_Notebooks/four_varieties/four_cropped_varieties_artigo_80_20/val_results3/weights/best.pt")

# Validation and Metrics

In [ ]:
from ultralytics import YOLO
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

def evaluate_model(
    model_path,
    dataset_path,
    project_path,
    experiment_name='validation_results'
):
    """
    Evaluates a YOLOv8 classification model on the test set.

    Parameters:
        model_path (str): Path to the trained model's .pt file.
        dataset_path (str): Path to the dataset directory (containing train, val, test subfolders) or the data.yaml file.
        project_path (str): Path to the directory where results will be saved.
        experiment_name (str): Name of the subdirectory to save the results (default: 'validation_results').

    Returns:
        metrics: Metrics object returned by the .val() method.
    """
    # Load the trained model
    best_model = YOLO(model_path)

    # Evaluate the model on the validation/test set
    # For classification, 'data' typically points to the dataset root or data.yaml
    metrics = best_model.val(
        data=dataset_path,
        project=project_path,
        name=experiment_name
    )

    return metrics

In [ ]:
# Metrics Visualization and Reporting

def metrics_model(
    metrics,
    plot_confusion_matrix=True
):
    """
    Processes and displays evaluation metrics from a YOLOv8 model.

    Parameters:
        metrics: Metrics object returned by the .val() method.
        plot_confusion_matrix (bool): If True, plots the graphical confusion matrix (default: True).
    """
    # Display main metrics
    print("Top-1 Accuracy:", metrics.results_dict['metrics/accuracy_top1'])
    print("Top-5 Accuracy:", metrics.results_dict['metrics/accuracy_top5'])
    print("Fitness:", metrics.results_dict['fitness'])

    # Display speeds
    print("\nSpeeds (ms per image):")
    print("  Preprocessing:", metrics.speed['preprocess'])
    print("  Inference:", metrics.speed['inference'])
    print("  Loss Calculation:", metrics.speed['loss'])
    print("  Postprocessing:", metrics.speed['postprocess'])

    # Ensure class_names are consistent with your dataset
    # These class names must match the order in your data.yaml or model.names
    class_names = ['CTC_1007', 'IAC_5503', 'RB_5375', 'RB_7515'] # Example: Update with actual class names

    # Display Confusion Matrix
    confusion_matrix = metrics.confusion_matrix.matrix
    confusion_matrix = confusion_matrix.astype(int)  # Convert to integers

    print("\nConfusion Matrix:")
    print(confusion_matrix)

    # Plot graphical confusion matrix (if requested)
    if plot_confusion_matrix:
        plt.figure(figsize=(8, 6))
        sns.set(font_scale=1.2)  # Adjust font size
        sns.heatmap(confusion_matrix, annot=True, fmt='d', cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names, cbar=False)

        # Configure title and axis labels
        plt.title('Confusion Matrix', fontsize=16)
        plt.xlabel('Predicted Class', fontsize=14)
        plt.ylabel('True Class', fontsize=14)

        # Show the plot
        plt.show()

    # Display additional information
    print("\nAdditional Information:")
    print(f"  Confidence Threshold (conf): {metrics.confusion_matrix.conf}")
    print(f"  IoU Threshold (iou_thres): {metrics.confusion_matrix.iou_thres}")
    print(f"  Number of Classes (nc): {metrics.confusion_matrix.nc}")
    print(f"  Plotting Enabled (plot): {metrics.confusion_matrix.plot}")
    print(f"  True Positives / False Positives (tp_fp): {metrics.confusion_matrix.tp_fp}")
    error_rate = 1 - metrics.results_dict['metrics/accuracy_top1']
    print("  Error Rate:", error_rate)

    # Check for curves
    if metrics.curves:
        print("\nAvailable curves:", metrics.curves)

## Validation Procedure (\val)

In [ ]:
# EXECUTING VALIDATION (\VAL)
# IMPORTANT: Update these paths to your trained model, dataset, and desired project output location.
model_path =   './runs/classify/my_classification_experiment/val_results/weights/best.pt' # Path to your best trained model
dataset_path = './datasets' # Path to your dataset directory (contains train, val, test folders)
project_path = './runs/classify/my_validation_project' # Project path for validation results
experiment_name = 'validation_run'

metrics = evaluate_model(model_path, dataset_path, project_path, experiment_name)

metrics_model(metrics=metrics)

In [ ]:
model.

## Test Set Validation Procedure (\test)

### 1. Special Handling for Test Set Validation: (Temporarily rename directories)

To evaluate the model specifically on the test set using YOLO's `val` mode, a common approach is to temporarily rename your `test` directory to `val` and, if applicable, your original `val` directory to `val_backup`. This 'tricks' YOLOv8 into using your test set as its validation set. **Please exercise caution when doing this and ensure you revert the directory names afterward to avoid data integrity issues.**

**Procedure:**
1.  **BEFORE RUNNING:** Temporarily rename your `val` directory (e.g., `datasets/val`) to `val_backup` (e.g., `datasets/val_backup`).
2.  **BEFORE RUNNING:** Temporarily rename your `test` directory (e.g., `datasets/test`) to `val` (e.g., `datasets/val`).
3.  **EXECUTE THE CODE CELL BELOW.**
4.  **AFTER RUNNING:** Revert the directory names: rename `datasets/val` back to `datasets/test` and `datasets/val_backup` back to `datasets/val`.

In [ ]:
#ANTES DE RODAR, FAZER:
#   1. ALTERAR O NOME DE \VAL PARA \VAL_BACKUP
#   2. ALTERAR O NOME DE \TEST PARA \VAL
#   3. EXECUTAR CÓDIGO



model_path = '/content/drive/MyDrive/Colab_Notebooks/four_varieties/four_cropped_varieties_artigo_80_20/val_results3/weights/best.pt'
dataset_path = '/content/drive/MyDrive/Colab_Notebooks/four_varieties/data.yaml'
project_path = '/content/drive/MyDrive/Colab_Notebooks/four_varieties/four_cropped_varieties_classif_30_epocas_80_20'
experiment_name = 'test_results'

metrics = evaluate_model(model_path, dataset_path, project_path, experiment_name)
metrics_model(metrics=metrics, dataset_path=dataset_path)

#ANTES DE RODAR, FAZER:
#   1. ALTERAR O NOME DE \VAL PARA \TEST
#   2. ALTERAR O NOME DE \VAL_BACKUP PARA \VAL


In [ ]:
print(model.names)

In [ ]:
from ultralytics import YOLO

# Path to the trained model weights
# IMPORTANT: Update this path to your trained model's .pt file.
model_path = './runs/classify/my_classification_experiment/val_results/weights/best.pt'

# Load the model
model = YOLO(model_path)

# Get class names from the model
class_names = model.names

# Display class names and their indices
print("Model Classes:")
for class_index, class_name in class_names.items():
    print(f"  Index: {class_index}, Name: {class_name}")

# Inferences

Code to Load the Model

Code to Test the Model on the Test Set

In [ ]:
model.predict()

In [ ]:
from ultralytics import YOLO
import os
import matplotlib.pyplot as plt
from PIL import Image

# Path to the trained model with the best weights
# IMPORTANT: Update this path to your trained model's 'best.pt' file.
best_model_path = "./runs/classify/my_classification_experiment/val_results/weights/best.pt"

# Load the model
model = YOLO(best_model_path)

# Path to the folder containing unseen images for inference
# IMPORTANT: Update this path to the directory with images you want to make predictions on.
# These images should not have been part of the training, validation, or test sets.
pasta_nunca_vistas = "./inference_images/cropped" # Example: './inference_images/cropped'

project = "./runs/classify/inference_project" # Project path to save inference results
name = "my_predictions" # Name for the inference run

# Perform predictions on all images in the specified folder
results = model.predict(source=pasta_nunca_vistas, save=True, project=project, name=name)

# Display results for each image
for i, result in enumerate(results):
    # Image file name
    image_filename = result.path  # Get the full image path
    print(f"Image {i + 1}: {os.path.basename(image_filename)}")  # Display only the file name

    # Check if it's a classification model result
    if hasattr(result, 'probs'):
        # Predicted class and probabilities
        predicted_class_index = result.probs.top1 # Get the index of the top predicted class
        predicted_class_name = model.names[predicted_class_index] # Get the name using model.names
        probabilities = result.probs.data.tolist()
        print(f"  Predicted Class: {predicted_class_name} (Index: {predicted_class_index})")
        print(f"  Probabilities: {probabilities}")
    else:
        # For detection or segmentation models (if applicable)
        boxes = result.boxes  # Bounding boxes
        print(f"  Number of detections: {len(boxes)}")

    # Visualize the image with predictions
    plt.figure(figsize=(6, 6))  # Set figure size
    plt.imshow(Image.open(image_filename))  # Display the original image
    plt.axis('off')  # Turn off axes
    plt.title(f"Class: {predicted_class_name if hasattr(result, 'probs') else 'Detection'}")  # Add a title
    plt.show()

In [ ]:
from ultralytics import YOLO
import os
import matplotlib.pyplot as plt
from PIL import Image

# Path to the trained model with the best weights
# IMPORTANT: Update this path to your trained model's 'best.pt' file.
best_model_path = "./runs/classify/my_classification_experiment/val_results/weights/best.pt"

# Load the model
model = YOLO(best_model_path)

# Path to the folder with unseen images or a specific subset for inference
# IMPORTANT: Update this path to the directory containing images for inference.
# For example, to run inference on the test set:
# pasta_nunca_vistas = './datasets/test'
# Or on a specific class within the test set:
pasta_nunca_vistas = './datasets/test/CTC_1007' # Example path

# Perform predictions on all images in the specified folder
# 'save=True' saves the results (e.g., predicted images) to the 'runs/classify/predict' directory.
# 'save_txt=True' saves prediction labels to .txt files alongside the predicted images.
results = model.predict(source=pasta_nunca_vistas, save=True, save_txt=True)

# Display results for each image
for i, result in enumerate(results):
    # Image file name
    image_filename = result.path  # Get the full image path
    print(f"Image {i + 1}: {os.path.basename(image_filename)}")  # Display only the file name

    # Predicted class and probabilities
    predicted_class_index = result.probs.top1 # Get the index of the top predicted class
    predicted_class_name = model.names[predicted_class_index] # Get the name using model.names
    print(f"  Predicted Class: {predicted_class_name} (Index: {predicted_class_index})")
    print(f"  Probabilities: {result.probs.data.tolist()}")

    # Visualize the image
    image = Image.open(image_filename)  # Open the image using PIL
    plt.figure(figsize=(6, 6))  # Set figure size
    plt.imshow(image)  # Display the image
    plt.axis('off')  # Turn off axes
    plt.title(f"Predicted Class: {predicted_class_name}")  # Add a title with the predicted class
    plt.show()

In [ ]:
# No loop de inferência:
target_layer = model.model.model[-2]  # Última camada convolucional (exemplo)
print(target_layer)

In [ ]:
from ultralytics import YOLO
import os
import matplotlib.pyplot as plt
from PIL import Image
import torch
import numpy as np
import cv2

def grad_cam(model, image_path, target_layer, class_idx=None):
    """
    Generates a Grad-CAM heatmap for a given image and model.

    Args:
        model (YOLO): The trained YOLO classification model.
        image_path (str): Path to the input image.
        target_layer (torch.nn.Module): The layer in the model to extract features from.
        class_idx (int, optional): The index of the target class. If None, the top predicted class is used.

    Returns:
        tuple: A tuple containing:
            - cam (np.ndarray): The Grad-CAM heatmap.
            - original_image (np.ndarray): The original image as a NumPy array.
    """
    # Load and preprocess the image
    image = Image.open(image_path).convert('RGB')
    # The model.transforms automatically applies necessary transformations (resize, normalize, ToTensor)
    image_tensor = model.transforms(image) # Applies model's default transforms
    image_tensor = image_tensor.unsqueeze(0).to(model.device)  # Add batch dimension and move to device

    # Prepare model to capture gradients
    model.eval()
    model.zero_grad()

    # Structures to store activations and gradients
    features = {}
    gradients = {}

    # Hook to capture activations and gradients
    def hook_fn(module, input, output):
        if isinstance(output, tuple):
            output = output[0]  # Handle tuple outputs
        output.requires_grad_(True)
        features['features'] = output
        output.register_hook(lambda grad: gradients.setdefault('gradients', grad))

    # Register the hook on the target layer
    handle = target_layer.register_forward_hook(hook_fn)

    # Forward pass
    outputs = model(image_tensor)
    probs = outputs[0].probs  # Get Probs object for classification
    logits = probs.data  # Extract probability tensor

    # Ensure correct dimensions (batch, classes)
    if logits.dim() == 1:
        logits = logits.unsqueeze(0)

    # Determine target class
    if class_idx is None:
        class_idx = logits.argmax().item() # Use the top predicted class if not specified

    # Create one-hot vector for the target class
    one_hot = torch.zeros_like(logits)
    one_hot[0, class_idx] = 1.0

    # Backward pass to compute gradients
    logits.backward(gradient=one_hot, retain_graph=True)

    # Process activations and gradients
    activations = features['features'].detach().cpu().numpy()[0]
    grads = gradients['gradients'].cpu().numpy()[0]

    # Calculate weights based on global average pooling of gradients
    weights = np.mean(grads, axis=(1, 2))
    cam = np.zeros(activations.shape[1:], dtype=np.float32)

    # Generate heatmap
    for i, w in enumerate(weights):
        cam += w * activations[i]

    # Post-process the heatmap
    cam = np.maximum(cam, 0) # Apply ReLU
    cam = cv2.resize(cam, image.size[::-1])  # Resize to original image dimensions (width, height)
    cam = cam / (cam.max() + 1e-8)  # Normalize to [0, 1]

    # Remove the hook to prevent memory leaks
    handle.remove()

    return cam, np.array(image)

# Function to visualize Grad-CAM
def visualize_grad_cam(image, heatmap, alpha=0.4):
    """
    Visualizes the Grad-CAM heatmap overlaid on the original image.

    Args:
        image (np.ndarray): Original image (NumPy array).
        heatmap (np.ndarray): Grad-CAM heatmap.
        alpha (float): Transparency of the heatmap overlay.

    Returns:
        np.ndarray: Image with the Grad-CAM overlay.
    """
    heatmap = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = (alpha * heatmap + (1 - alpha) * image).astype(np.uint8)
    return overlay

# Path to the trained model with the best weights
# IMPORTANT: Update this path to your trained model's 'best.pt' file.
best_model_path = "./runs/classify/my_classification_experiment/val_results/weights/best.pt"

# Load the model
model = YOLO(best_model_path)

# Path to the folder with unseen images for inference
# IMPORTANT: Update this path to the directory with images you want to analyze with Grad-CAM.
pasta_nunca_vistas = "./inference_images/cropped" # Example: './inference_images/cropped'
project = "./runs/classify/gradcam_results" # Project path to save Grad-CAM related output
name = "gradcam_predictions" # Name for the Grad-CAM inference run

# Perform predictions on all images in the specified folder
# 'save=True' here will save the prediction results (e.g., original images with labels)
results = model.predict(source=pasta_nunca_vistas, save=True, project=project, name=name)

# Display results and Grad-CAM for each image
for i, result in enumerate(results):
    # Image file name
    image_filename = result.path  # Get the full image path
    print(f"Image {i + 1}: {os.path.basename(image_filename)}")  # Display only the file name

    # Check if it's a classification model result
    if hasattr(result, 'probs'):
        # Predicted class and probabilities
        predicted_class_index = result.probs.top1 # Get the index of the top predicted class
        predicted_class_name = model.names[predicted_class_index] # Get the name using model.names
        probabilities = result.probs.data.tolist()
        print(f"  Predicted Class: {predicted_class_name} (Index: {predicted_class_index})")
        print(f"  Probabilities: {probabilities}")

        # Apply Grad-CAM
        # Typically, the target layer is the last convolutional layer before the classification head.
        # model.model.model[-2] is a common choice for YOLOv8 classification models.
        target_layer = model.model.model[-2] # Last convolutional layer (adjust as necessary)
        heatmap, original_image = grad_cam(
            model,
            image_filename,
            target_layer,
            class_idx=predicted_class_index) # Pass the predicted class index for targeted CAM

        # Visualize Grad-CAM
        overlay = visualize_grad_cam(original_image, heatmap)

        # Display the original image and the Grad-CAM overlay
        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        plt.imshow(Image.open(image_filename))
        plt.title("Original Image")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.imshow(overlay)
        plt.title(f"Grad-CAM (Class: {predicted_class_name})")
        plt.axis('off')

        plt.show()
    else:
        # For detection or segmentation models, display detections (if applicable)
        boxes = result.boxes  # Bounding boxes
        print(f"  Number of detections: {len(boxes)}")

        # Visualize the image with predictions
        plt.figure(figsize=(6, 6))  # Set figure size
        plt.imshow(Image.open(image_filename))  # Display the original image
        plt.axis('off')  # Turn off axes
        plt.title(f"Detection ({len(boxes)} objects)")  # Add a title
        plt.show()

# Export for Production
Enables export to ONNX or TensorFlow Lite

In [ ]:
# Exportar para ONNX ou TensorFlow Lite
model.export(format='onnx')  # ONNX é compatível com muitos frameworks
model.export(format='tflite')  # TensorFlow Lite é compatível com Android e iOS
